# SAP Cloud ALM CDM Features API

This notebook demonstrates the core operations of the SAP Cloud ALM **CDM Features API** (`calm-features/v1`):

- Authentication (OAuth2 Client Credentials or sandbox API key)
- Reference data: Feature statuses and priorities
- Querying features with OData options (`$filter`, `$select`, `$expand`, `$count`)
- Creating a new Feature
- Updating a Feature (PATCH)
- Pagination with `@odata.nextLink`

## Prerequisites

**Environment setup (one-time):**
1. Create a virtual environment: `python -m venv .venv`
2. Activate it: `source .venv/bin/activate` (macOS/Linux) or `.venv\Scripts\activate` (Windows)
3. Install dependencies: `pip install -r requirements.txt`

## Sandbox vs. Production

Set `USE_SANDBOX = True` in the setup cell to run against the public **SAP API Business Hub sandbox** — no tenant credentials required, read-only.

Set `USE_SANDBOX = False` to use your own SAP Cloud ALM tenant with OAuth2. Create an `apidata.py` file from `apidata_template.py`:

```python
# apidata.py
cdm_features_client_id     = 'your client ID'
cdm_features_client_secret = 'your client secret'
token_url = 'https://<identityzone>.authentication.<region>.hana.ondemand.com/oauth/token'
base_url  = 'https://<tenant>.<region>.alm.cloud.sap'
```

**Never commit `apidata.py`** — it contains credentials.

The **create and update cells (Sections 5–6)** require production OAuth credentials — the sandbox does not support write operations.

In [ ]:
!python -m pip install -q requests
print("Dependencies ready")

## 1. Setup and Authentication

Set `USE_SANDBOX` and (if using sandbox) `SANDBOX_APIKEY` before running.

The cell defines two header dicts:
- `HEADERS` — used for all GET requests
- `JSON_HEADERS` — adds `Content-Type: application/json` for POST and PATCH requests

In [ ]:
import requests
from urllib.parse import urljoin
import json

# ── Toggle ────────────────────────────────────────────────────────────────────
USE_SANDBOX    = True   # True = sandbox (read-only, no credentials); False = production (OAuth2)
SANDBOX_APIKEY = ""     # Required when USE_SANDBOX = True — get a free key at https://api.sap.com

FEATURES_API_PATH = "/api/calm-features/v1/"
SANDBOX_BASE      = "https://sandbox.api.sap.com/SAPCALM/calm-features/v1/"

def _features_api_root(base_url: str) -> str:
    """Normalize any form of base_url to the calm-features/v1/ root."""
    root = base_url.strip().rstrip("/")
    for marker in ("/api/calm-features/v1/Features", "/api/calm-features/v1"):
        if marker.lower() in root.lower():
            root = root[: root.lower().find(marker.lower())]
            break
    return root.rstrip("/") + FEATURES_API_PATH

def get_token(token_url, client_id, client_secret):
    r = requests.post(
        token_url,
        data={"grant_type": "client_credentials"},
        auth=(client_id, client_secret),
        timeout=30,
    )
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        print(e.response.text)
        raise
    return r.json()["access_token"]

if USE_SANDBOX:
    if not SANDBOX_APIKEY.strip():
        raise ValueError("Set SANDBOX_APIKEY to your API key from https://api.sap.com")
    BASE_URL = SANDBOX_BASE
    HEADERS  = {"apikey": SANDBOX_APIKEY}
    print(f"Mode: SANDBOX (read-only)  Base URL: {BASE_URL}")
else:
    import apidata as ad
    token    = get_token(ad.token_url, ad.cdm_features_client_id, ad.cdm_features_client_secret)
    # BTP tokens typically expire after 12 hours — re-run this cell if subsequent calls return 401
    BASE_URL = _features_api_root(ad.base_url)
    HEADERS  = {"Authorization": f"Bearer {token}"}
    print(f"Mode: PRODUCTION  Base URL: {BASE_URL}")

JSON_HEADERS = {**HEADERS, "Content-Type": "application/json"}

## 2. Finding Your Project ID

Most CDM Features API calls require a `projectId` (UUID). Set it directly below:

- **Default demo project:** `11111111-1111-1111-1111-111111111111` — this project exists in every SAP Cloud ALM tenant (including the sandbox) and is safe to use for testing.
- **Your own project:** copy the UUID from the SAP Cloud ALM UI (it appears in the project URL).

If you leave `PROJECT_ID` empty, the cell discovers candidate project IDs from existing Features via
`GET /Features?$select=projectId` (no additional API scope required).

In [ ]:
from uuid import UUID

# Option A (recommended): paste project UUID from SAP Cloud ALM UI URL.
# The default value below exists in every CALM tenant (including the sandbox) and is safe for demos.
PROJECT_ID = "11111111-1111-1111-1111-111111111111"

# Option B: leave PROJECT_ID empty to auto-pick from feature-derived candidates.

def _normalize_uuid(value: str) -> str:
    return str(UUID(value.strip().strip("'").strip('"')))

def _discover_project_ids_from_features(max_rows: int = 200) -> list:
    features_url = urljoin(BASE_URL, "Features")
    r = requests.get(
        features_url, headers=HEADERS,
        params={"$select": "projectId", "$top": str(max_rows)},
        timeout=60,
    )
    r.raise_for_status()
    seen, ids = set(), []
    for item in r.json().get("value", []):
        pid = item.get("projectId")
        if pid and pid not in seen:
            seen.add(pid)
            ids.append(pid)
    return sorted(ids)

if PROJECT_ID.strip():
    PROJECT_ID = _normalize_uuid(PROJECT_ID)
    print(f"Using provided PROJECT_ID: {PROJECT_ID}")
    candidates = _discover_project_ids_from_features()
    if candidates and PROJECT_ID not in candidates:
        print(
            "Warning: Provided PROJECT_ID was not seen in feature-derived candidates. "
            "Proceeding anyway — the project may be new (no features yet)."
        )
    if candidates:
        print("\nCandidate project IDs discovered from existing features:")
        for pid in candidates[:20]:
            print(f"  {pid}")
        if len(candidates) > 20:
            print(f"  ... and {len(candidates) - 20} more")
else:
    candidates = _discover_project_ids_from_features()
    if candidates:
        PROJECT_ID = candidates[0]
        print(f"Using first discovered projectId: {PROJECT_ID}")
        print("\nAll discovered project IDs:")
        for pid in candidates[:20]:
            print(f"  {pid}")
    else:
        raise ValueError(
            "PROJECT_ID is empty and no feature-derived candidates were found. "
            "Paste PROJECT_ID from SAP Cloud ALM UI URL."
        )

## 3. Reference Data

The API exposes two read-only reference entities:
- **FeatureStatus** — valid status codes and their labels (e.g., `CREATED`, `IN_REALIZATION`)
- **FeaturePriorities** — valid priority codes (`10` = Very High … `40` = Low)

Use these to validate inputs before creating or patching Features.

> **Note:** `APPROVED_FOR_DEPLOYMENT` is a valid status but cannot be set via the API — it is reserved for internal deployment workflows.

In [ ]:
for entity in ("FeatureStatus", "FeaturePriorities"):
    r = requests.get(urljoin(BASE_URL, entity), headers=HEADERS, timeout=30)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        print(e.response.text)
        raise
    items = r.json().get("value", [])
    print(f"\n{entity} ({len(items)} items):")
    for item in items:
        print(f"  {item}")

## 4. List Features

### 4a. Basic list with `$top` and `$orderby`

The sandbox reliably supports `$orderby=modifiedAt desc`. For production tenants a fallback loop
tries multiple sort fields because some tenants return HTTP 400 for certain `$orderby` values.

In [ ]:
features_url = urljoin(BASE_URL, "Features")

if USE_SANDBOX:
    params = {"$top": 5, "$orderby": "modifiedAt desc"}
    r = requests.get(features_url, headers=HEADERS, params=params, timeout=60)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        print(e.response.text)
        raise
    data = r.json()
else:
    orderby_candidates = ["modifiedAt", "createdAt", "displayId", "title", "statusCode", "priorityCode"]
    data = None
    for orderby in orderby_candidates:
        r = requests.get(features_url, headers=HEADERS, params={"$top": 5, "$orderby": orderby}, timeout=60)
        try:
            r.raise_for_status()
            data = r.json()
            print(f"Using $orderby={orderby!r}")
            break
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 400:
                continue
            print(e.response.text if e.response is not None else str(e))
            raise
    if data is None:
        r = requests.get(features_url, headers=HEADERS, params={"$top": 5}, timeout=60)
        r.raise_for_status()
        data = r.json()
        print("No supported $orderby — fell back to $top only")

print(f"Returned {len(data.get('value', []))} feature(s)")
print(json.dumps(data.get("value", [])[:1], indent=2))

### 4b. Filter by project, status, and priority

OData `$filter` rules:
- String values must be **single-quoted**: `statusCode eq 'IN_REALIZATION'`
- Integer comparisons use no quotes: `priorityCode le 20`
- UUID values (`Edm.Guid`) must be **unquoted**: `projectId eq 11111111-1111-1111-1111-111111111111`
- Do **not** quote `projectId` — comparing `Edm.Guid` to `Edm.String` returns HTTP 400
- Always include a deterministic `$orderby` when paging

Set `PROJECT_ID` in **Section 2** before running this cell.

In [ ]:
from urllib.parse import urlencode, quote

def _odata_get(url: str, params: dict):
    # Use percent-encoding for spaces (%20) instead of '+' for OData expression compatibility.
    query = urlencode(params, quote_via=quote)
    full_url = f"{url}?{query}"
    return requests.get(full_url, headers=HEADERS, timeout=60), full_url

# Normalize UUID — Edm.Guid filter must be unquoted
project_uuid = str(UUID(PROJECT_ID.strip().strip("'")))
features_url = urljoin(BASE_URL, "Features")
base_params  = {
    "$filter": f"projectId eq {project_uuid}",
    # Combined example: f"projectId eq {project_uuid} and statusCode eq 'IN_REALIZATION' and priorityCode le 20"
    "$top": "50",
}

if USE_SANDBOX:
    r, full_url = _odata_get(features_url, {**base_params, "$orderby": "modifiedAt desc"})
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        print(e.response.text)
        raise
    data = r.json()
    used_orderby = "modifiedAt desc"
else:
    orderby_candidates = ["modifiedAt", "createdAt", "displayId", "title", "statusCode", "priorityCode"]
    data, used_orderby, full_url = None, None, None
    for orderby in orderby_candidates:
        r, full_url = _odata_get(features_url, {**base_params, "$orderby": orderby})
        try:
            r.raise_for_status()
            data = r.json()
            used_orderby = orderby
            break
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 400:
                continue
            print(e.response.text if e.response is not None else str(e))
            raise
    if data is None:
        r, full_url = _odata_get(features_url, base_params)
        r.raise_for_status()
        data = r.json()
        print("$orderby rejected for all tested fields; executed filter without $orderby.")

if used_orderby:
    print(f"Using $orderby={used_orderby!r}")
print(f"Using $filter=projectId eq {project_uuid}")
print(f"Matched {len(data.get('value', []))} feature(s)")
print(json.dumps(data.get("value", [])[:1], indent=2))

### 4c. Project fields with `$select` and expand related entities with `$expand`

`$expand` resolves navigation properties inline. Available expansions:
`toStatus`, `toPriority`, `toProject`, `toRelease`, `toScope`, `toWorkstream`,
`toURLReferences`, `toExternalReferences`

> **Note:** Projects, Releases, Scopes and Workstreams have no top-level endpoints — they are
> only accessible via `$expand` on Features.

In [ ]:
features_url = urljoin(BASE_URL, "Features")
base_params  = {
    "$select": "uuid,displayId,title,projectId,statusCode,priorityCode,modifiedAt",
    "$expand": "toStatus,toPriority,toProject,toScope,toRelease,toWorkstream,toURLReferences,toExternalReferences",
    "$top": "3",
}

if USE_SANDBOX:
    r = requests.get(features_url, headers=HEADERS, params={**base_params, "$orderby": "modifiedAt desc"}, timeout=60)
    r.raise_for_status()
    data = r.json()
else:
    orderby_candidates = ["modifiedAt", "createdAt", "displayId", "title", "statusCode", "priorityCode"]
    data = None
    for orderby in orderby_candidates:
        r = requests.get(features_url, headers=HEADERS, params={**base_params, "$orderby": orderby}, timeout=60)
        try:
            r.raise_for_status()
            data = r.json()
            print(f"Using $orderby={orderby!r}")
            break
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 400:
                continue
            print(e.response.text if e.response is not None else str(e))
            raise
    if data is None:
        r = requests.get(features_url, headers=HEADERS, params=base_params, timeout=60)
        r.raise_for_status()
        data = r.json()

print(json.dumps(data.get("value", [])[:1], indent=2))

## 5. Create a Feature

Minimum required fields: `title` and `projectId`.

Use `PROJECT_ID` from **Section 2**. You can also include nested `toURLReferences` and
`toExternalReferences` arrays in the same POST request.

Valid status codes: `CREATED`, `NOT_PLANNED`, `IN_REALIZATION`, `IN_TESTING`, `SUCCESSFULLY_TESTED`, `CONFIRMED`  
Valid priority codes: `10` (Very High), `20` (High), `30` (Medium), `40` (Low)

> **Sandbox:** The sandbox does not support write operations. This cell is skipped when `USE_SANDBOX = True`.

In [ ]:
if USE_SANDBOX:
    CREATED_UUID = None
    print(
        "Sandbox mode is read-only: POST /Features is not allowed. "
        "Set USE_SANDBOX = False and configure OAuth credentials in apidata.py to create features."
    )
else:
    FEATURE_TYPE = ""       # Optional. Example: "CALMFEAT"
    WORKSTREAM_ID = ""      # Optional. Example: "NTOP"
    INCLUDE_REFERENCES_ON_CREATE = False  # Some tenants reject nested refs on create.

    payload = {
        "title": "API Demo Feature",
        "projectId": PROJECT_ID,
        "description": "Created via the CDM Features API notebook",
        "statusCode": "CREATED",
        "priorityCode": 30,
    }
    if FEATURE_TYPE.strip():
        payload["type"] = FEATURE_TYPE.strip()
    if WORKSTREAM_ID.strip():
        payload["workstreamId"] = WORKSTREAM_ID.strip()
    if INCLUDE_REFERENCES_ON_CREATE:
        payload["toURLReferences"] = [{"name": "Specification", "url": "https://example.com/spec"}]
        payload["toExternalReferences"] = [
            {"id": "jira:ABC-123", "name": "Jira ABC-123", "url": "https://jira.example.com/browse/ABC-123"}
        ]

    create_url = urljoin(BASE_URL, "Features")
    print(f"POST {create_url}")
    print(json.dumps(payload, indent=2))

    r = requests.post(create_url, headers=JSON_HEADERS, json=payload, timeout=60)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        body = e.response.text if e.response is not None else str(e)
        print(f"HTTP {e.response.status_code if e.response is not None else 'n/a'}")
        print(body)
        if "doesn't exist" in body and "Project" in body:
            print("Hint: choose a projectId from the Section 2 candidate list, or paste one from SAP Cloud ALM UI URL.")
        raise

    created = r.json()
    CREATED_UUID = created.get("uuid")
    if not CREATED_UUID:
        raise KeyError(f"Response has no 'uuid'. Response keys: {list(created.keys())}")
    print(f"Created feature UUID: {CREATED_UUID}")
    print(json.dumps(created, indent=2))

## 6. Update a Feature (PATCH)

PATCH only sends the fields you want to change — other fields are left untouched.

> **Important:** Do not set `APPROVED_FOR_DEPLOYMENT` via the API — that status is reserved for
> internal deployment workflows and will return a 400 error.

> **Note:** The CDM Features API does not provide a DELETE endpoint. To remove a feature created
> during testing, use the SAP Cloud ALM UI.

> **Sandbox:** The sandbox does not support write operations. This cell is skipped when `USE_SANDBOX = True`.

In [ ]:
if USE_SANDBOX:
    print(
        "Sandbox mode is read-only: PATCH /Features/{uuid} is not allowed. "
        "Set USE_SANDBOX = False and configure OAuth credentials in apidata.py to update features."
    )
else:
    # Uses CREATED_UUID from the previous cell; set manually if running independently.
    FEATURE_UUID = CREATED_UUID  # or: FEATURE_UUID = "01234567-89ab-cdef-0123-456789abcdef"
    if not FEATURE_UUID:
        raise ValueError("No feature UUID available. Create or provide a feature UUID before PATCH.")

    patch_body = {
        "statusCode": "IN_REALIZATION",
        "description": "Implementation in progress",
    }
    r = requests.patch(
        urljoin(BASE_URL, f"Features/{FEATURE_UUID}"),
        headers=JSON_HEADERS,
        json=patch_body,
        timeout=60,
    )
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        print(e.response.text)
        raise
    print(f"PATCH successful (HTTP {r.status_code})")

    r = requests.get(
        urljoin(BASE_URL, f"Features/{FEATURE_UUID}"),
        headers=HEADERS,
        params={"$expand": "toStatus,toPriority"},
        timeout=60,
    )
    r.raise_for_status()
    print(json.dumps(r.json(), indent=2))

## 7. Pagination

The recommended pagination approach for the Features API is **`@odata.nextLink`** — follow the
server-provided next-page URL until no further link is returned. This avoids the duplicate-record
risk of offset-based `$skip` paging on live data.

Always pair pagination with a deterministic `$orderby` to get stable, non-overlapping pages.

In [ ]:
PAGE_SIZE    = 5
MAX_PAGES    = 3  # limit for this demo
features_url = urljoin(BASE_URL, "Features")
base_params  = {"$top": str(PAGE_SIZE), "$count": "true"}

if USE_SANDBOX:
    r = requests.get(features_url, headers=HEADERS, params={**base_params, "$orderby": "modifiedAt desc"}, timeout=60)
    r.raise_for_status()
    first_body = r.json()
else:
    orderby_candidates = ["modifiedAt", "createdAt", "displayId", "title", "statusCode", "priorityCode"]
    first_body = None
    for orderby in orderby_candidates:
        r = requests.get(features_url, headers=HEADERS, params={**base_params, "$orderby": orderby}, timeout=60)
        try:
            r.raise_for_status()
            first_body = r.json()
            print(f"Using $orderby={orderby!r}")
            break
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 400:
                continue
            print(e.response.text if e.response is not None else str(e))
            raise
    if first_body is None:
        r = requests.get(features_url, headers=HEADERS, params=base_params, timeout=60)
        r.raise_for_status()
        first_body = r.json()
        print("No supported $orderby — fell back to $top/$count only")

all_features = []
page = 1
print(f"Total count: {first_body.get('@odata.count', 'n/a')}")
batch = first_body.get("value", [])
all_features.extend(batch)
print(f"Page 1: {len(batch)} feature(s) fetched (running total: {len(all_features)})")
next_url = first_body.get("@odata.nextLink")

while next_url and page < MAX_PAGES:
    r = requests.get(next_url, headers=HEADERS, timeout=60)
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        print(e.response.text if e.response is not None else str(e))
        raise
    body = r.json()
    batch = body.get("value", [])
    all_features.extend(batch)
    page += 1
    print(f"Page {page}: {len(batch)} feature(s) fetched (running total: {len(all_features)})")
    next_url = body.get("@odata.nextLink")

print(f"\nFetched {len(all_features)} feature(s) across {page} page(s)")